In [1]:
# # farpoint Algorithm on MNIST
import os
import random
import pickle
import math
import itertools
import time

from bidict import bidict
from datetime import datetime
import numpy as np
from scipy.spatial import distance
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from sklearn.manifold import TSNE
from sklearn import datasets, metrics

from farpoint_base import Auto_Oracle, COPK_Engine, Partition, NQ_Engine
# from farpoint_base_post_algo import NQ_Engine
from farpoint_testing import skew_weighted_accuracy

import MNIST_skew

In [2]:
%matplotlib qt

In [3]:
# HYPERPARAMETERS
MAX_NUM_QUERIES = 500
NUM_QUERIES_PER_ITERATION = 1


# choose one or the other
SPLIT_METHOD = "NQ" # Nearest_Queried
# SPLIT_METHOD = "COPKM" #'COPK_Means'
# SPLIT_METHOD = "HA" #'Single_Link'

# choose one or the other
# SOURCE = 'human'
SOURCE = 'auto' # get answers directly from pre-existing labels

# hoping for repeatable behavior
# np.random.seed(20) # DON'T USE OVER MULTIPLE COMPUTER RUNS


In [4]:
# DATA BLOCK

# NOTE: Assumption for FP_Slim is that user will have data in desired form ready to go
# e.g. skewing and decimation have already happened if necessary

# this all eventually is part of "umbrella" - skewed would go in this data class
NUM_CLASSES = MNIST_skew.NUM_CLASSES

whos_computer = 'Ro'
X,y = MNIST_skew.get_MNIST(whos_computer)
X,y,_,_ = MNIST_skew.skew_data(X, y)
print(X.shape,y.shape)


(15735, 784) (15735,)


In [5]:
# DECIMATION REMOVED - SLOW............................................

# # Decimation?
# DECIMATION_FACTOR = 10
# X = X[0:-1:DECIMATION_FACTOR,:]
# y = y[0:-1:DECIMATION_FACTOR]
# print(X.shape)
# print(y.shape)

# # Dimensionality Reduction?

In [6]:
oracle = Auto_Oracle(y, NUM_QUERIES_PER_ITERATION)
if SPLIT_METHOD == "COPKM":
    clusterer = COPK_Engine()
elif SPLIT_METHOD == "NQ":
    clusterer = NQ_Engine() 
data_partition = Partition(clusterer)

for iter_num in range(MAX_NUM_QUERIES//NUM_QUERIES_PER_ITERATION): 
    oracle.reset() # happens each iter
    data_partition.run(X,oracle)
    predicted_labels = data_partition.predict_labels(X)[0]
    translated_predictions = np.vectorize(data_partition.label_key_.inv.get)(predicted_labels).astype(int)
    if iter_num*NUM_QUERIES_PER_ITERATION%10 == 0:
        print(f'number of queries = {iter_num*NUM_QUERIES_PER_ITERATION}, \
            number of classes = {len(data_partition.label_key_)}')




number of queries = 0,             number of classes = 1
number of queries = 10,             number of classes = 4
number of queries = 20,             number of classes = 5
number of queries = 30,             number of classes = 6
number of queries = 40,             number of classes = 7
number of queries = 50,             number of classes = 8
number of queries = 60,             number of classes = 8
number of queries = 70,             number of classes = 8
number of queries = 80,             number of classes = 8
number of queries = 90,             number of classes = 9
number of queries = 100,             number of classes = 9
number of queries = 110,             number of classes = 9
number of queries = 120,             number of classes = 9
number of queries = 130,             number of classes = 9
number of queries = 140,             number of classes = 9
number of queries = 150,             number of classes = 9
number of queries = 160,             number of classes = 10
number 

In [7]:
# n = 700
# print(y[n],translated_predictions[n])
# plt.imshow(X[n].reshape((28,28)))

In [8]:
print(data_partition.label_key_)

bidict({'0': 0, '6': 1, '4': 2, '3': 3, '7': 4, '2': 5, '1': 6, '8': 7, '9': 8, '5': 9})


In [9]:
all_pts_labels,l_pts_labels, o_pts_labels = data_partition.predict_labels(X)
# for MNIST, since we just treated the ys as integers, need to translate these, since
# partition mapped them first-come,first-served
all_pts_labels_tr = np.vectorize(data_partition.label_key_.inv.get)(all_pts_labels).astype(int)
# can't translate l_pts_labels,o_pts_labels due to -1's, so translate as needed individually later

In [10]:
# migrate this to module visualization.py once tested
## t-SNE
# from sklearn.manifold import TSNE
# X_2d = TSNE(n_components=2).fit_transform(X)
# X_2d.shape

## PCA - fast but not good (for visualization here)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca.fit(X)
X_2d = pca.transform(X)

In [11]:
# took from seaborn
palette =[(0.00784313725490196, 0.24313725490196078, 1.0), \
                (1.0, 0.48627450980392156, 0.0), \
                (0.10196078431372549, 0.788235294117647, 0.2196078431372549), \
                (0.9098039215686274, 0.0, 0.043137254901960784), \
                (0.5450980392156862, 0.16862745098039217, 0.8862745098039215),\
                (0.6235294117647059, 0.2823529411764706, 0.0), \
                (0.9450980392156862, 0.2980392156862745, 0.7568627450980392),\
                (0.6392156862745098, 0.6392156862745098, 0.6392156862745098),\
                (1.0, 0.7686274509803922, 0.0), (0.0, 0.8431372549019608, 1.0)]

In [12]:
# This is scatterplot "Raw w/ L-pts'
l_pts_fig = plt.figure()

plt.plot(X_2d[:,0], X_2d[:,1], marker = 'o', linestyle = 'none', markerfacecolor = (0,0.1,0.5), \
         markeredgecolor = (1,1,1), alpha = 0.5,markersize = 5)

for n in np.unique(l_pts_labels):
    # these l-pt labels are the integer labels internal to the partition - and must be translated
    if n != -1:
        this_n_indices = np.where(l_pts_labels == n)[0]
        n_tr = int(data_partition.label_key_.inv.get(n))
        plt.plot(X_2d[this_n_indices,0], X_2d[this_n_indices,1], marker = 'o', \
                 linestyle = 'none', markeredgecolor = (1,1,1), markerfacecolor = palette[n_tr], \
                 alpha = 1,markersize = 10,label = str(n_tr))

plt.legend()
plt.grid()
plt.title('Raw Data w/ L-pts')

Text(0.5, 1.0, 'Raw Data w/ L-pts')

In [13]:
print(y)
plt.figure()
plt.imshow(X[-1,:].reshape((28,28)))

[5 5 5 ... 1 1 1]


In [14]:
# This is scatterplot 'Actual'
true_labels_fig = plt.figure()
for n in np.unique(y):
        this_n_indices = np.where(y == n)[0]
        plt.plot(X_2d[this_n_indices,0], X_2d[this_n_indices,1], marker = 'o', \
                 linestyle = 'none', markeredgecolor = (1,1,1), markerfacecolor = palette[n], \
                 alpha = 0.5,markersize = 10,label = str(n))
plt.legend()
plt.grid()
plt.title('Actual')

Text(0.5, 1.0, 'Actual')

In [15]:
predicted_labels = data_partition.predict_labels(X)[0]


plt.figure()
for n in np.unique(predicted_labels):
    # these l-pt labels are the integer labels internal to the partition - and must be translated
    this_n_indices = np.where(predicted_labels == n)[0]
    n_tr = int(data_partition.label_key_.inv.get(n))
    plt.plot(X_2d[this_n_indices,0], X_2d[this_n_indices,1], marker = 'o', \
             linestyle = 'none', markeredgecolor = (1,1,1), markerfacecolor = palette[n_tr], \
             alpha = 1,markersize = 10,label = str(n_tr))
plt.legend()
plt.grid()
plt.title('Predicted')

Text(0.5, 1.0, 'Predicted')

In [16]:
# get accuracy stats
# must translate
translated_predictions = np.vectorize(data_partition.label_key_.inv.get)(predicted_labels).astype(int)
num_matches = np.where(translated_predictions == y)[0].shape[0]
size = y.shape[0]
print()
print("accuracy is: ,", num_matches / size)
from sklearn.metrics import balanced_accuracy_score
b_acc = balanced_accuracy_score(y, translated_predictions)
print("balanced accuracy is: ,", b_acc)

# it should be noted that accuracy is: 1) including known labels from oracle, and 2) not skewed accuracy


accuracy is: , 0.7265967588179219
balanced accuracy is: , 0.6051048855297623


In [17]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
conf_matrix = confusion_matrix(y, translated_predictions)

fig, ax = plt.subplots(figsize=(8,6), dpi=100)

# initialize using the raw 2D confusion matrix 
# and output labels (in our case, it's 0 and 1)
display = ConfusionMatrixDisplay(conf_matrix, display_labels=np.array(range(10)))

# set the plot title using the axes object
ax.set(title='Confusion Matrix for Farpoint')

# show the plot. 
# Pass the parameter ax to show customizations (ex. title) 
display.plot(ax=ax,values_format='');

In [18]:
from sklearn.metrics import recall_score
rs = recall_score(y, translated_predictions, average= None)
print(rs)

[0.83130081 0.58867589 0.42622951 0.89126016 0.82275267 0.4
 0.94692737 0.53658537 0.4        0.20731707]


In [19]:
count_list = []
for n in range(10):
    inds = np.where(y == n)[0]
    count_list.append(len(inds))
print(count_list)
    

[492, 7877, 61, 984, 1969, 15, 3938, 123, 30, 246]
